Search  Engine With Tools and Agents

In [26]:
import os 
from dotenv import load_dotenv

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "Search_Engine_Tool_Agents"

In [27]:
##Arxiv--Research
##Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [28]:
##used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [29]:
##used the inbuilt tool of arxiv
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=3,doc_content_chars_max=2000)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [30]:
tools=[wiki,arxiv]

In [31]:
##custom tools [RAG tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [32]:
loader=WebBaseLoader("https://docs.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A083C74D60>, search_kwargs={})

Convert retriever → tool

Because agents can only call tools, not raw retrievers.

In [33]:
from langchain_core.tools import tool

@tool
def langsmith_search(query: str) -> str:
    """Search any information about LangSmith."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [34]:
tools=[wiki,arxiv,langsmith_search]

In [35]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\GENAI_PROJECTS\\Search_Engine_using_Tools_Agents\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=2000)),
 StructuredTool(name='langsmith_search', description='Search any information about LangSmith.', args_schema=<class 'langchain_core.utils.pydantic.langsmith_search'>, func=<function langsmith_search at 0x000002A083B36950>)]

In [36]:
##Run all the tools with Agents and LLM models

#tools+llm model-->agent Executor
llm=ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant"
)

In [37]:
##prompt template 
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent")

In [38]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [39]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.messages[0].prompt.template
)

In [40]:
agent.invoke(
    {"messages": [("user", "What is LangSmith?")]}
)

{'messages': [HumanMessage(content='What is LangSmith?', additional_kwargs={}, response_metadata={}, id='a83ca083-fe3f-43e2-b994-ffa5a7f67abc'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '15dwv1zxj', 'function': {'arguments': '{"query":"What is LangSmith?"}', 'name': 'langsmith_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 493, 'total_tokens': 512, 'completion_time': 0.025208214, 'completion_tokens_details': None, 'prompt_time': 0.123103593, 'prompt_tokens_details': None, 'queue_time': 0.046740167, 'total_time': 0.148311807}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ce5f5-71e5-7363-8ffb-351ac5c6db8a-0', tool_calls=[{'name': 'langsmith_search', 'args': {'query': 'What is LangSmith?'}, 'id': '15dwv1zxj', 'type': 'tool_call'}], invalid_tool_calls=[],

In [41]:
result = agent.invoke(
    {"messages": [("user", "How to use LangSmith?")]}
)
final_answer = result["messages"][-1].content
print(final_answer)

Unfortunately, the information provided was cut off. To find more information about how to use LangSmith, you can try searching the internet or looking at the official documentation.


In [42]:
result = agent.invoke(
    {"messages": [("user", "What research papers exist about large language models?")]}
)
final_answer = result["messages"][-1].content
print(final_answer)

It appears that the search results from the ArXiv and LangSmith functions have been interrupted. To get the full results, I will try to call the functions again.

 <function=arxiv>{"query": "large language models research papers 2025"}</function>
 <function=arxiv>{"query": "Large Language Models and their applications in 2025"}</function>
 <function=langsmith_search>{"query": "large language models large language models and applications"}</function>
